In [ ]:
import hashlib
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
from Bio import SeqIO

In [ ]:
MUESTREO_XLSX = "muestreo_PAIRED_89.xlsx"
RUNS = list(pd.read_excel(MUESTREO_XLSX, sheet_name="Muestra_PAIRED_89")["run_accession"])

THREADS          = 12
FRACCION_SECCION = 0.10
N_SECCIONES      = 2
SEED             = 100

CARD_URL             = "https://card.mcmaster.ca/latest/data"
AMRFINDERPLUS_DB_DIR = Path("localDB/amrfinderplus")
AMRFINDER_DB         = AMRFINDERPLUS_DB_DIR / "latest"
BLASTDB_DIR          = Path("localDB/blastdb")
NT_DB                = BLASTDB_DIR / "nt"
TAXID_BACTERIA       = "2"

print(f"{len(RUNS)} muestras a procesar ({MUESTREO_XLSX})")

In [ ]:
def sh(cmd):
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        el = time.time() - t0
        print(f"[{int(el // 60):>2}m{int(el % 60):02d}s] {line}", end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Fallo (codigo {proc.returncode}): {cmd}")


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}TB"


def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def reintentar(fn, intentos=5, espera_s=5, descripcion="operacion"):
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            return fn()
        except Exception as e:
            ultimo_error = e
            if intento < intentos:
                print(f"  Fallo en {descripcion} ({e}); reintentando en {espera_s}s [{intento}/{intentos}]...")
                time.sleep(espera_s)
    raise RuntimeError(f"'{descripcion}' fallo tras {intentos} intentos: {ultimo_error}") from ultimo_error


def _consultar_ena_filereport(url, descripcion):
    def _get():
        df = pd.read_csv(url, sep="\t")
        if df.empty or "fastq_ftp" not in df.columns or pd.isna(df.loc[0, "fastq_ftp"]):
            raise ValueError(f"respuesta de ENA sin fastq_ftp utilizable: {df.to_dict('records')[:1]}")
        return df
    return reintentar(_get, descripcion=descripcion)

In [ ]:
if not Path("localDB/card.json").exists():
    if not Path("card.json").exists():
        sh(f"wget -q {CARD_URL} -O card_data.tar.bz2")
        sh("tar -xjf card_data.tar.bz2 ./card.json")
    sh("rgi load --card_json card.json --local")
assert Path("localDB/card.json").exists(), "CARD no quedo cargada"

if not AMRFINDER_DB.exists():
    AMRFINDERPLUS_DB_DIR.mkdir(parents=True, exist_ok=True)
    sh(f"amrfinder_update -d {AMRFINDERPLUS_DB_DIR}")
assert AMRFINDER_DB.exists(), "AMRFinderPlus DB no quedo lista"

BLASTDB_DIR.mkdir(parents=True, exist_ok=True)
if not ((BLASTDB_DIR / "nt.nal").exists() or (BLASTDB_DIR / "nt.nin").exists()):
    sh(f"cd {BLASTDB_DIR} && update_blastdb.pl --decompress --source ncbi nt")
if not (BLASTDB_DIR / "taxdb.btd").exists():
    sh(f"cd {BLASTDB_DIR} && update_blastdb.pl --decompress taxdb")
os.environ["BLASTDB"] = str(BLASTDB_DIR.resolve())

In [ ]:
def descargar_con_aria2c(pares_url_nombre, dest_dir, threads):
    input_file = dest_dir / "aria2_input.txt"
    with open(input_file, "w") as f:
        for url, nombre in pares_url_nombre:
            f.write(f"{url}\n  out={nombre}\n")
    conexiones = min(threads, 16)
    sh(f"aria2c -i {input_file} -d {dest_dir} -x{conexiones} -s{conexiones} -j2 -c "
       f"--max-tries=0 --retry-wait=5 --timeout=120 --allow-overwrite=true --quiet=true")


def resolver_y_descargar(run, raw_dir):
    ena_url = (
        "https://www.ebi.ac.uk/ena/portal/api/filereport"
        f"?accession={run}&result=read_run&fields=fastq_ftp,fastq_md5&format=tsv"
    )
    ena = _consultar_ena_filereport(ena_url, descripcion=f"consulta ENA ({run})")
    ftp = ena.loc[0, "fastq_ftp"].split(";")
    md5_esperado = ena.loc[0, "fastq_md5"].split(";")

    r1, r2 = raw_dir / f"{run}_1.fastq.gz", raw_dir / f"{run}_2.fastq.gz"
    if not all(r.exists() and md5sum(r) == m for r, m in zip((r1, r2), md5_esperado)):
        descargar_con_aria2c([("https://" + ftp[0], r1.name), ("https://" + ftp[1], r2.name)], raw_dir, THREADS)
        for archivo, esperado in zip((r1, r2), md5_esperado):
            obtenido = md5sum(archivo)
            if obtenido != esperado:
                raise RuntimeError(f"MD5 no coincide para {archivo.name}: {obtenido} != {esperado}")
    print(f"Descarga verificada (MD5): {r1.name}, {r2.name}")
    return r1, r2

In [ ]:
LINEAS_POR_LECTURA = 4


def qc_y_submuestreo(run, r1, r2, work_dir, out_dir):
    clean1, clean2 = work_dir / "clean_1.fastq.gz", work_dir / "clean_2.fastq.gz"
    sh(f"fastp -i {r1} -I {r2} -o {clean1} -O {clean2} -w {THREADS} "
       f"-h {out_dir}/{run}_fastp.html -j {out_dir}/{run}_fastp.json")

    frac_total = N_SECCIONES * FRACCION_SECCION
    pool1, pool2 = work_dir / "pool_1.fastq", work_dir / "pool_2.fastq"
    sh(f"seqtk sample -s{SEED} {clean1} {frac_total} > {pool1}")
    sh(f"seqtk sample -s{SEED} {clean2} {frac_total} > {pool2}")

    pool_lineas = int(subprocess.run(f"wc -l < {pool1}", shell=True, text=True, capture_output=True).stdout)
    n_seccion = pool_lineas // LINEAS_POR_LECTURA // N_SECCIONES

    sub_pares = {}
    for i in range(1, N_SECCIONES + 1):
        ini_linea, n_lineas = (i - 1) * n_seccion * LINEAS_POR_LECTURA, n_seccion * LINEAS_POR_LECTURA
        sub1, sub2 = work_dir / f"sub_1_{i}.fastq", work_dir / f"sub_2_{i}.fastq"
        sh(f"tail -n +{ini_linea + 1} {pool1} | head -n {n_lineas} > {sub1}")
        sh(f"tail -n +{ini_linea + 1} {pool2} | head -n {n_lineas} > {sub2}")
        sub_pares[i] = (sub1, sub2)

    pool1.unlink()
    pool2.unlink()
    return sub_pares


def ensamblar_y_predecir_genes(sub1, sub2, work_dir, i):
    megahit_out = work_dir / f"megahit_out_{i}"
    shutil.rmtree(megahit_out, ignore_errors=True)
    sh(f"megahit -1 {sub1} -2 {sub2} -t {THREADS} -o {megahit_out}")
    contigs = megahit_out / "final.contigs.fa"

    faa, fna, gff = work_dir / f"genes_{i}.faa", work_dir / f"genes_{i}.fna", work_dir / f"genes_{i}.gff"
    sh(f"prodigal -i {contigs} -a {faa} -d {fna} -p meta -q -o {gff} -f gff")
    return contigs, faa, fna, gff


def detectar_arg(run, i, contigs, faa, gff, out_dir):
    rgi_out = out_dir / f"rgi_{run}_{i}"
    sh(f"rgi main -i {faa} -o {rgi_out} -t protein -a DIAMOND --local --clean")
    rgi = pd.read_csv(f"{rgi_out}.txt", sep="\t")

    amr_out = out_dir / f"amrfinder_{run}_{i}.tsv"
    sh(f"amrfinder -p {faa} -n {contigs} -g {gff} -a prodigal "
       f"-d {AMRFINDER_DB} -o {amr_out} --threads {THREADS}")
    amrfinder = pd.read_csv(amr_out, sep="\t")

    print(f"Seccion {i}: {len(rgi)} ARG (CARD/RGI), {len(amrfinder)} ARG (AMRFinderPlus)")
    return rgi, amrfinder

In [ ]:
def normalizar_gen(nombre):
    return re.sub(r"[^A-Z0-9]", "", str(nombre).upper())


def _partes_gen(nombre):
    n = normalizar_gen(nombre)
    m = re.match(r"^([A-Z]*)([0-9].*)?$", n)
    return m.group(1), m.group(2) or ""


def mismo_gen(a, b, min_len=3):
    pa, na = _partes_gen(a)
    pb, nb = _partes_gen(b)
    if not pa or not pb:
        return False
    if na != nb:
        return False
    if len(pa) < min_len or len(pb) < min_len:
        return pa == pb
    return pa in pb or pb in pa


def cruzar_card_amrfinder(rgi, amrfinder):
    genes_card = sorted(rgi["Best_Hit_ARO"].dropna().unique())
    genes_amr = sorted(amrfinder["Element symbol"].dropna().unique())

    filas, vistos_amr = [], set()
    for g_card in genes_card:
        candidatos = [g for g in genes_amr if mismo_gen(g_card, g)]
        vistos_amr.update(candidatos)
        filas.append({"gen": g_card, "en_CARD": True, "en_AMRFinderPlus": bool(candidatos),
                      "gen_AMRFinderPlus": ", ".join(candidatos) or None})
    for g_amr in genes_amr:
        if g_amr not in vistos_amr:
            filas.append({"gen": g_amr, "en_CARD": False, "en_AMRFinderPlus": True, "gen_AMRFinderPlus": g_amr})
    cruce_gen = pd.DataFrame(filas, columns=["gen", "en_CARD", "en_AMRFinderPlus", "gen_AMRFinderPlus"])

    familias_card = sorted(rgi["Drug Class"].dropna().str.split(";").explode().str.strip().unique())
    clases_amr = sorted(amrfinder["Class"].dropna().unique())

    filas_fam, vistas_amr = [], set()
    for f_card in familias_card:
        candidatas = [c for c in clases_amr if mismo_gen(f_card, c)]
        vistas_amr.update(candidatas)
        filas_fam.append({"familia": f_card, "en_CARD": True, "en_AMRFinderPlus": bool(candidatas)})
    for c_amr in clases_amr:
        if c_amr not in vistas_amr:
            filas_fam.append({"familia": c_amr, "en_CARD": False, "en_AMRFinderPlus": True})
    cruce_familia = pd.DataFrame(filas_fam, columns=["familia", "en_CARD", "en_AMRFinderPlus"])

    return cruce_gen, cruce_familia


def contig_de_orf(orf, asm_ids):
    orf = str(orf).split()[0]
    if orf in asm_ids:
        return orf
    partes = orf.split("_")
    for corte in range(len(partes) - 1, 0, -1):
        cand = "_".join(partes[:corte])
        if cand in asm_ids:
            return cand
    return None


def coords_desde_header_prodigal(orf_id):
    m = re.search(r"#\s*(\d+)\s*#\s*(\d+)\s*#\s*(-?1)\s*#", str(orf_id))
    if not m:
        return pd.Series([None, None, None])
    inicio, fin, hebra = m.groups()
    return pd.Series([int(inicio), int(fin), int(hebra)])


def agregar_posicion_relativa(df, contigs_path, col_contig, col_start, col_stop, resolver_contig=None):
    largos = {r.id: len(r.seq) for r in SeqIO.parse(str(contigs_path), "fasta")}
    df = df.copy()
    if resolver_contig:
        asm_ids = set(largos)
        df["contig_asm"] = df[col_contig].map(lambda x: resolver_contig(x, asm_ids))
    else:
        df["contig_asm"] = df[col_contig]
    df["contig_len"] = df["contig_asm"].map(largos)
    df["pos_inicio_rel"] = df[col_start] / df["contig_len"]
    df["pos_fin_rel"] = df[col_stop] / df["contig_len"]
    return df


def agregar_secuencias(df, faa_path, fna_path, col_id):
    proteinas = {r.id: str(r.seq) for r in SeqIO.parse(str(faa_path), "fasta")}
    nucleotidos = {r.id: str(r.seq) for r in SeqIO.parse(str(fna_path), "fasta")}
    df = df.copy()
    ids = df[col_id].map(lambda x: str(x).split()[0])
    df["secuencia_aa"] = ids.map(proteinas)
    df["secuencia_nt"] = ids.map(nucleotidos)
    return df

In [ ]:
_BLAST_COLS = ["qseqid", "sacc", "slen", "qstart", "qend", "sstart", "send", "pident", "length", "stitle"]


def organismo_de_descripcion(desc):
    desc = desc.replace("[", "").replace("]", "")
    desc = re.sub(r"^(PREDICTED:|UNVERIFIED:|MAG:|TPA:|TPA_asm:)\s*", "", desc).strip()
    palabras = desc.split()
    if not palabras:
        return None
    if palabras[0].lower() in ("uncultured", "unidentified", "bacterium", "synthetic"):
        return palabras[0].lower()
    if len(palabras) >= 2 and palabras[1][0].islower():
        return f"{palabras[0]} {palabras[1]}"
    return palabras[0]


def blast_local_taxonomia_y_posicion(contigs_recs, out_dir, tag):
    vacio = {"taxon": None, "identidad_%": None, "descripcion": "sin hits", "hit_accession": None,
             "hit_len_ref": None, "query_start": None, "query_end": None,
             "sbjct_start": None, "sbjct_end": None}
    if not contigs_recs:
        return pd.DataFrame(columns=["contig", *vacio.keys()])

    query_fa = out_dir / f"_blast_query_{tag}.fasta"
    out_tsv = out_dir / f"_blast_out_{tag}.tsv"
    with open(query_fa, "w") as f:
        for r in contigs_recs:
            f.write(f">{r.id}\n{r.seq}\n")

    sh(f"blastn -query {query_fa} -db {NT_DB} -task megablast -taxids {TAXID_BACTERIA} "
       f"-max_target_seqs 1 -max_hsps 1 -outfmt \"6 {' '.join(_BLAST_COLS)}\" "
       f"-num_threads {THREADS} -out {out_tsv}")

    if out_tsv.exists() and out_tsv.stat().st_size > 0:
        hits = (pd.read_csv(out_tsv, sep="\t", names=_BLAST_COLS)
                 .groupby("qseqid", as_index=False).first().set_index("qseqid"))
    else:
        hits = pd.DataFrame(columns=_BLAST_COLS).set_index("qseqid")
    query_fa.unlink(missing_ok=True)
    out_tsv.unlink(missing_ok=True)

    filas = []
    for r in contigs_recs:
        if r.id not in hits.index:
            filas.append({"contig": r.id, **vacio})
            continue
        h = hits.loc[r.id]
        filas.append({"contig": r.id, "taxon": organismo_de_descripcion(str(h["stitle"])),
                      "identidad_%": round(float(h["pident"]), 1), "descripcion": str(h["stitle"])[:70],
                      "hit_accession": h["sacc"], "hit_len_ref": int(h["slen"]),
                      "query_start": int(h["qstart"]), "query_end": int(h["qend"]),
                      "sbjct_start": int(h["sstart"]), "sbjct_end": int(h["send"])})
    return pd.DataFrame(filas)


def proyectar_posicion_en_referencia(row):
    qs, qe = row.get("query_start"), row.get("query_end")
    ss, se = row.get("sbjct_start"), row.get("sbjct_end")
    inicio, fin = row.get("Start"), row.get("Stop")
    if pd.isna(qs) or pd.isna(ss) or pd.isna(inicio) or pd.isna(fin):
        return pd.Series({"ref_pos_inicio": None, "ref_pos_fin": None})
    if not (qs <= inicio <= qe and qs <= fin <= qe):
        return pd.Series({"ref_pos_inicio": None, "ref_pos_fin": None})  # ARG fuera del tramo alineado

    def proyectar(pos):
        frac = (pos - qs) / (qe - qs) if qe != qs else 0
        return round(ss + frac * (se - ss))

    return pd.Series({"ref_pos_inicio": proyectar(inicio), "ref_pos_fin": proyectar(fin)})


def vincular_arg_a_especie(rgi, amrfinder, contigs, out_dir, tag):
    contigs_por_id = {r.id: r for r in SeqIO.parse(str(contigs), "fasta")}
    wanted = set(rgi["contig_asm"].dropna()) | set(amrfinder["contig_asm"].dropna())
    recs = [contigs_por_id[c] for c in wanted if c in contigs_por_id]
    blast_df = blast_local_taxonomia_y_posicion(recs, out_dir, tag)

    def enriquecer(df):
        df = df.merge(blast_df, left_on="contig_asm", right_on="contig", how="left").drop(columns="contig")
        if len(df):
            df[["ref_pos_inicio", "ref_pos_fin"]] = df.apply(proyectar_posicion_en_referencia, axis=1)
        else:
            df["ref_pos_inicio"], df["ref_pos_fin"] = None, None
        return df

    return enriquecer(rgi), enriquecer(amrfinder)

In [ ]:
def limpiar_muestra(run, raw_dir, work_dir):
    liberado = 0
    for ruta in (raw_dir, work_dir):
        if not ruta.exists():
            continue
        liberado += sum(f.stat().st_size for f in ruta.rglob("*") if f.is_file())
        shutil.rmtree(ruta, ignore_errors=True)
    print(f"Limpieza: liberados {_human(liberado)} de {run}")


def procesar_muestra(run):
    raw_dir, work_dir, out_dir = Path(f"raw/{run}"), Path(f"work/{run}"), Path(f"results/{run}")
    resistoma_out = out_dir / f"resistoma_{run}.csv"
    if resistoma_out.exists():
        print(f"{run}: ya procesada, se omite")
        return pd.read_csv(resistoma_out)

    print(f"\n{'=' * 60}\nProcesando muestra: {run}\n{'=' * 60}")
    shutil.rmtree(work_dir, ignore_errors=True)
    for d in (raw_dir, work_dir, out_dir):
        d.mkdir(parents=True, exist_ok=True)

    r1, r2 = resolver_y_descargar(run, raw_dir)
    sub_pares = qc_y_submuestreo(run, r1, r2, work_dir, out_dir)

    secciones = []
    for i, (sub1, sub2) in sub_pares.items():
        contigs, faa, fna, gff = ensamblar_y_predecir_genes(sub1, sub2, work_dir, i)
        rgi, amr = detectar_arg(run, i, contigs, faa, gff, out_dir)

        cruce_gen, cruce_familia = cruzar_card_amrfinder(rgi, amr)
        cruce_gen.to_csv(out_dir / f"cruce_gen_{run}_{i}.csv", index=False)
        cruce_familia.to_csv(out_dir / f"cruce_familia_{run}_{i}.csv", index=False)

        if len(rgi):
            # rgi main -t protein deja Start/Stop/Orientation vacios -- se sacan del header de prodigal en ORF_ID
            rgi[["Start", "Stop", "Orientation"]] = rgi["ORF_ID"].apply(coords_desde_header_prodigal)
        rgi = agregar_posicion_relativa(rgi, contigs, "ORF_ID", "Start", "Stop", resolver_contig=contig_de_orf)
        amr = agregar_posicion_relativa(amr, contigs, "Contig id", "Start", "Stop")

        rgi = agregar_secuencias(rgi, faa, fna, "ORF_ID")
        amr = agregar_secuencias(amr, faa, fna, "Protein id")

        rgi, amr = vincular_arg_a_especie(rgi, amr, contigs, out_dir, f"{run}_{i}")

        rgi.insert(0, "herramienta", "CARD/RGI")
        amr.insert(0, "herramienta", "AMRFinderPlus")
        seccion = pd.concat([rgi, amr], ignore_index=True)
        seccion.insert(0, "seccion", i)
        secciones.append(seccion)

    resistoma = pd.concat(secciones, ignore_index=True)
    resistoma.insert(0, "run", run)
    resistoma.to_csv(resistoma_out, index=False)

    limpiar_muestra(run, raw_dir, work_dir)
    return resistoma

In [ ]:
resultados_por_muestra, fallas = {}, {}
for run in RUNS:
    try:
        resultados_por_muestra[run] = procesar_muestra(run)
    except Exception as e:
        print(f"\n!! {run} fallo, se continua con la siguiente muestra: {type(e).__name__}: {e}\n")
        fallas[run] = e
        shutil.rmtree(Path(f"work/{run}"), ignore_errors=True)

print(f"\nMuestras procesadas: {len(resultados_por_muestra)}/{len(RUNS)}")
if fallas:
    print(f"Muestras que fallaron ({len(fallas)}): {list(fallas)}")

In [ ]:
resultados_totales = (pd.concat(resultados_por_muestra.values(), ignore_index=True)
                      if resultados_por_muestra else pd.DataFrame())
print(f"Genes de resistencia detectados, todas las muestras (CARD + AMRFinderPlus): {len(resultados_totales)}")

card = resultados_totales[resultados_totales["herramienta"] == "CARD/RGI"] if len(resultados_totales) else resultados_totales
tabla_arg = (
    card.groupby(["Best_Hit_ARO", "AMR Gene Family", "Drug Class"])
    .agg(n_detecciones=("run", "size"), n_muestras=("run", "nunique"))
    .sort_values(["n_muestras", "n_detecciones"], ascending=False)
    .reset_index()
) if len(card) else pd.DataFrame()
tabla_arg